# Model Training (Keras)

## Library

In [1]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_NVML_BASED_CUDA_CHECK'] = '1'
os.environ['KERAS_BACKEND'] = 'torch'
import torch

print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
  print('Device Name:', torch.cuda.get_device_name(0))
import keras
from keras.models import Sequential, load_model
from keras.layers import Input, Dense, Activation, Dropout, LeakyReLU
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras_tuner import HyperModel, RandomSearch

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import numpy as np
import pandas as pd
import random
from parameters import *
from pathlib import Path
home = Path.home()

GPU Available: True
Device Name: NVIDIA A100 80GB PCIe MIG 4g.40gb


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Activation, Dropout, LeakyReLU
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os
import numpy as np
import pandas as pd

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras_tuner import HyperModel, RandomSearch
from tensorflow.keras.models import load_model

# Disable some console warnings
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


#validating GPUs with tf
from tensorflow.python.client import device_lib
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

GPUs Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
#using pytorch
'''seed = 0
random.seed(seed)
np.random.seed(seed)

# PyTorch (covers CPU and all GPU devices like your A100)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Optional: ensure deterministic operations for strict reproducibility (may slow down training slightly)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False'''

'seed = 0\nrandom.seed(seed)\nnp.random.seed(seed)\n\n# PyTorch (covers CPU and all GPU devices like your A100)\ntorch.manual_seed(seed)\ntorch.cuda.manual_seed_all(seed)\n\n# Optional: ensure deterministic operations for strict reproducibility (may slow down training slightly)\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.benchmark = False'

In [5]:
#using tensorflow
seed = 0

# Input seed value. if this value is the same, the random number generator will generate the same set of random values every time
np.random.seed(seed)

# Set the seed value for reproducibility in tensorflow
tf.random.set_seed(seed)

## Dataset

### Load

In [6]:
# Keeping DATA_AUGMENTATION 'True' for now so everything scales between 0 and 1
if DATA_AUGMENTATION:
    x_train_val = np.load('{}/npy/y_{}_train_val_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    x_test = np.load('{}/npy/y_{}_test_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True) 

    Y_train_val = np.load('{}/npy/x_{}_train_val_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    Y_test = np.load('{}/npy/x_{}_test_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    
    Y_train_val_log = np.load('{}/npy/x_{}_train_val_augmented_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    Y_test_log = np.load('{}/npy/x_{}_test_augmented_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
else:
    x_train_val = np.load('{}/npy/y_{}_train_val.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    x_test = np.load('{}/npy/y_{}_test.npy'.format(DATA_DIR, PARAM), allow_pickle=True)

    Y_train_val = np.load('{}/npy/x_{}_train_val.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    Y_test = np.load('{}/npy/x_{}_test.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    
    Y_train_val_log = np.load('{}/npy/x_{}_train_val_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    Y_test_log = np.load('{}/npy/x_{}_test_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)

### Visualize

In [7]:
print(x_train_val.shape)
print(Y_train_val.shape[1])
#print(Y_train_val[6613])

(4312, 7)
3


In [8]:
#number of values in the training set and test set, these are probably pulling from the data files individual points

total = len(Y_train_val) + len(Y_test)
print('---------------------------------------')  
print('Train-and-validation set shape: {}, {:.2f}%'.format(len(Y_train_val), (len(Y_train_val)*100.)/total))
print('Test set shape:                 {}, {:.2f}%'.format(len(Y_test), (len(Y_test)*100.)/total))
print('---------------------------------------')

---------------------------------------
Train-and-validation set shape: 4312, 79.99%
Test set shape:                 1079, 20.01%
---------------------------------------


In [9]:
%matplotlib inline
import matplotlib.pyplot as plt

In [10]:
print(Y_train_val.shape, Y_train_val.dtype)
print(Y_train_val_log.shape, Y_train_val_log.dtype)

(4312, 3) object
(4312, 3) object


### Prepare

In [11]:
# Only grab drain current for now, because we have a fixed set of biases. 
# Combine the linear and log scale data (linear first, then log) into the same target dataset

lin_log_Y_train = []
for i in range(0,len(Y_train_val)):
    flattened = Y_train_val[i][2]
    logId = Y_train_val_log[i][2]
    flattened.extend(logId)
    lin_log_Y_train.append(np.asarray(flattened))

lin_log_Y_test = []
for i in range(0,len(Y_test)):
    flattened2 = Y_test[i][2]
    logId2 = Y_test_log[i][2]
    flattened2.extend(logId2)
    lin_log_Y_test.append(np.asarray(flattened2))

print(lin_log_Y_test[0].shape)

(370,)


## MLP

### Create model

Create a classical multi-layer perceptron for regression: 602 input features, 10 targets, and a certain number of hidden layers and neurons. Taking some inspiration from [Deep learning-based I-V Global Parameter Extraction for BSIM-CMG](https://www.sciencedirect.com/science/article/pii/S003811012300179X), Solid-State Electronics, Vol. 209, November 2023.

Reccomended to download a third party app like "Sleep control Center" or "Amphetamine" to prevent computer from sleeping during the many hour/day long training process

### Create Model by Hand

In [29]:
if KERAS_TUNER == False:
    
    # Multilayer perceptron (MLP) with 601 input features. MLP is having neurons that adjust rules based on how accurate they can guess things
    model_shape = 'mlp_602_'
    
    # Define the number on neurons in the inner layer (in parameter file)
    model_shape += '_'.join(str(l) for l in NEURONS_PER_LAYER)
    
    # 10 output neurons for 10 parameters
    model_shape += '_10'

In [30]:
if KERAS_TUNER == False:
    # Initialize a sequential model, which lets us build a linear stack of layers
    model = Sequential()
    
    # Add an input layer to the model. This layer defines the shape of the input data (602 features per sample), sets dimensions of following layers
    model.add(Input(shape=(370,), name='input1'))
    
    # Iterate over the configuration of neurons for each hidden layer specified in NEURONS_PER_LAYER
    for i, n in enumerate(NEURONS_PER_LAYER):
        # Add a fully connected (dense) hidden layer with spec ified number of neurons
        # The LeCun uniform initializer is used when initializing weights, this makes the model more stable
        # L2 regularization is used in each layer to penalizing large weights, which prevents overfitting
        model.add(Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    
        # Apply a Leaky ReLU activation function to the outputs of the dense layer
        # This introduces non-linearities, allowing the network to learn complex functions
        # Leaky ReLU is chosen over standard ReLU to help mitigate the "dying ReLU" problem:
        #     - This problem is when neurons using the ReLU activation function output zero for all inputs and stop learning
        #     - Can be mitigated by using variations like Leaky ReLU or proper initialization
        model.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
        
        # Add a dropout layer to reduce overfitting -- randomly drops a set fraction (like 30%) of outputs from the layer
        model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
    
    # Add the output layer consisting of 10 neurons, corresponding to the 10 target variables we aim to predict.
    # The same LeCun uniform initializer is used to ensure consistency and stability at the output layer as well.
    model.add(Dense(7, name='fc{}'.format(len(NEURONS_PER_LAYER)), kernel_initializer='lecun_uniform'))
    

In [31]:
if KERAS_TUNER == False:
    # The exponential decay learning rate schedule gradually reduces the learning rate, fine-tuning the learning process for better convergence
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR_INITIAL,  
        decay_steps=LR_DECAY_STEPS,        
        decay_rate=LR_DECAY_RATE,          
        staircase=LR_STAIRCASE             
    )
    
    # Set model to minimize loss specified by TRAIN_LOSS, and also to report the loss during training
    model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
        loss=TRAIN_LOSS,                                         
        metrics=[TRAIN_LOSS]                                     
    )

In [32]:
if KERAS_TUNER == False:
    !mkdir -p model
    best_model_file = 'model/{}_best_model.h5'.format(model_shape)
    last_model_file = 'model/{}_last_model.h5'.format(model_shape)

Enable training (`train_and_save`) to overwrite the model file.

In [12]:
train_and_save = True

We use Adam optimizer, minimize the Mean Squared Logarithmic Error, and early stop.

#### Training

In [34]:
from IPython.display import clear_output
if KERAS_TUNER == False:
    class TrainingPlot(tf.keras.callbacks.Callback):
         
        # This function is called when the training begins
        def on_train_begin(self, logs={}):
            # Initialize the lists for holding the logs, losses 
            self.losses = []
            self.val_losses = []
            self.logs = []
        
        # This function is called at the end of each epoch
        def on_epoch_end(self, epoch, logs={}):
            
            # Append the logs, losses  to the lists
            self.logs.append(logs)
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))
            
            # Before plotting ensure at least 2 epochs have passed
            if len(self.losses) > 1:
                
                # Clear the previous plot
                clear_output(wait=True)
                N = np.arange(0, len(self.losses))
                
                # Plot train loss, train acc, val loss and val acc against epochs passed
                plt.figure()
                plt.plot(N, self.losses, label = "train_loss")
                plt.plot(N, self.val_losses, label = "val_loss")
                plt.title("Training Loss [Epoch {}]".format(epoch))
                plt.xlabel("Epoch #")
                plt.ylabel("Loss/Accuracy")
                plt.legend()
                plt.show()
           
    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def __init__(self):
            super(LearningRateMonitor, self).__init__()
            self.learning_rates = []
    
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer._current_learning_rate.read_value()
            self.learning_rates.append(lr)
            print(f"Epoch: {epoch + 1}, Learning Rate: {lr:.10f}")

In [13]:
print(KERAS_TUNER)

True


In [85]:
if KERAS_TUNER == False:
    
    history = None  
    if train_and_save: 
        # Set up early stopping to prevent overfitting by halting training when validation loss stops improving
        early_stopping = EarlyStopping(
            monitor='val_loss',                      # Monitor validation loss for stopping criteria 
            mode='min',                              # Stop when the monitored quantity has stopped decreasing
            patience=TRAIN_EARLY_STOPPING_PATIENCE,  # Number of epochs to wait after last improvement
            verbose=1                                # Enable logging when early stopping happens
        )
    
        # Set up model checkpointing to save the model at its best validation loss:
        model_checkpoint = ModelCheckpoint(
            filepath=best_model_file,          
            monitor='val_loss',            # Save the model based on validation loss improvement
            mode='min',                    # Favor lower validation loss values for saving (minimize)
            save_best_only=True,           # Save only when validation loss improves
            verbose=0                      # No logging for model saving
        )
        
        plot_callback = TrainingPlot()      # Plot training progress
        lr_monitor = LearningRateMonitor()  # Watch learning rate changes
    
        # Train the model on the training data and validate on a portion of it
        history = model.fit(
            np.asarray(lin_log_X_train),  
            np.asarray(y_train_val),      
            epochs=100,                   
            batch_size=TRAIN_BATCH_SIZE,  
            validation_split=TRAIN_VALIDATION_SPLIT,  
            callbacks=[],#[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
            verbose=1  # Enable logging of the training process.
        )
        
        model.save(last_model_file)  # Save the final model when done training!


Load the saved best model and use it from now on.

In [86]:
if KERAS_TUNER == False:
    model = load_model(best_model_file, custom_objects={})

### Keras Tuner to Find Best Hyperparameters and Model

Run this if you want to use keras tuner to make the model rather than doing it by hand

In [87]:
#using pytorch/keras
'''from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Dense, Input, Dropout
from keras.models import Sequential, load_model
from keras.regularizers import l2
from keras_tuner import HyperModel, RandomSearch'''

'from keras.callbacks import EarlyStopping, ModelCheckpoint\nfrom keras.layers import Dense, Input, Dropout\nfrom keras.models import Sequential, load_model\nfrom keras.regularizers import l2\nfrom keras_tuner import HyperModel, RandomSearch'

In [14]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras_tuner import HyperModel, RandomSearch
from tensorflow.keras.models import load_model

In [15]:
#print(Y_train_val.min(), Y_train_val.max())   # log space: ~ -12 .. -3.  linear: 1e-12 .. 1e-3


In [84]:
if KERAS_TUNER:
    def build_hypermodel(hp):
        
        # Hyperparameters to tune
        if FOUR_LAYERS:
            layer_num=4
        else:
            layer_num=3
            
        #neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=100, max_value=5000, step=100) for i in range(layer_num)]
        neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32) for i in range(layer_num)]
                   # neurons = hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
        if TUNER_DROPOUT:
            dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)
        
        # Create Model in the same way that we do by hand
        model = Sequential()
        #model.add(Input(shape=(370,), name='input1'))
        model.add(Input(shape=(7,), name='input1'))
    
        for i, n in enumerate(neurons_per_layer):
            model.add(Dense(n, name=f'fc{i}', kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(0.001)))
            model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            if TUNER_DROPOUT:
                model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
    
        #model.add(Dense(7, name='output', kernel_initializer='lecun_uniform'))
        model.add(Dense(370, name='output', kernel_initializer='lecun_uniform'))
    
        # Learning rate configuration
        lr_initial = hp.Float('learning_rate', 1e-6, 5e-3, sampling='LOG', default=0.0001)
        #lr_initial = hp.Float('learning_rate', 5e-5, 5e-4, sampling='LOG', default=1e-4)
        #lr_initial = hp.Float('learning_rate', 1e-4, 1e-1, sampling='LOG', default=0.0001)
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr_initial,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE
        )
    
        model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_schedule), 
                      loss='mean_squared_error',
                      metrics=['mean_squared_error'])
                    #loss='mean_absolute_error',
                    #metrics = ['mean_absolute_error'])
        
        return model
    
    def build_hypermodel_with_limits(hp):
        # Hyperparameters to tune
        neurons_per_layer = []
        total_parameters = 0
        max_parameters = 8000 #12000  # Minimum total number of parameters required

        if FOUR_LAYERS:
            layer_num=4
        else:
            layer_num=3
            
        # Try generating layers with sufficient total parameters
        for i in range(layer_num):
            #neurons = hp.Int(f'neurons_{i}', min_value=100, max_value=5000, step=100)
            neurons = hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
            # Add neurons for the current layer
            neurons_per_layer.append(neurons)
    
        # Calculate the total number of parameters for the configuration
        input_size = 602  # Input shape defined
        total_parameters += input_size * neurons_per_layer[0] + neurons_per_layer[0]  # First layer
    
        for i in range(1, len(neurons_per_layer)):
            total_parameters += neurons_per_layer[i - 1] * neurons_per_layer[i] + neurons_per_layer[i]
    
        total_parameters += neurons_per_layer[-1] * 10 + 10  # Output layer with 10 neurons
    
        # Skip configurations where the total parameters are below the threshold
        if total_parameters < max_parameters:
            return None
        
        # Create Model
        model = Sequential()
        model.add(Input(shape=(602,), name='input1'))
        if TUNER_DROPOUT:
            dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)
            
        for i, n in enumerate(neurons_per_layer):
            model.add(Dense(n, name=f'fc{i}', kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(0.001)))
            model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            if TUNER_DROPOUT:
                model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
    
        model.add(Dense(10, name='output', kernel_initializer='lecun_uniform'))
    
        # Learning rate configuration
        lr_initial = hp.Float('learning_rate', 1e-7, 5e-3, sampling='LOG', default=0.0001)
        #lr_initial = hp.Float('learning_rate', 1e-6, 1e-1, sampling='LOG', default=0.0001)
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr_initial,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE
        )
    
        model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_schedule), 
                      loss='mean_squared_error',
                      metrics=['mean_squared_error'])
        return model

In [85]:
#Tusing pytorch/keras
'''if KERAS_TUNER: #USING TORCH

  def build_hypermodel(hp):
    # Hyperparameters to tune
    if FOUR_LAYERS:
      layer_num = 4
    else:
      layer_num = 3

    neurons_per_layer = [
        hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
        for i in range(layer_num)
    ]
    l2_rate = 1e-5 #hp.Float('l2', 1e-7, 1e-4, sampling='LOG')
    if TUNER_DROPOUT:
      dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)

    # Create Model
    model = Sequential()
    model.add(Input(shape=(7,), name='input1'))

    for i, n in enumerate(neurons_per_layer):
      model.add(
          Dense(
              n,
              name=f'fc{i}',
              kernel_initializer='lecun_uniform',
              kernel_regularizer=l2(l2_rate),#0.001),
          )
      )
      model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
      if TUNER_DROPOUT:
        model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))

    model.add(Dense(370, name='output', kernel_initializer='lecun_uniform'))

    # Learning rate configuration
    lr_initial = hp.Float(
        'learning_rate', 1e-6, 5e-3, sampling='LOG', default=0.0001
    )
    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr_initial,
        decay_steps=LR_DECAY_STEPS,
        decay_rate=LR_DECAY_RATE,
        staircase=LR_STAIRCASE,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mean_squared_error',
        metrics=['mean_squared_error'],
        #loss='mean_absolute_error',
        #metrics = ['mean_absolute_error'],
    )

    return model

  def build_hypermodel_with_limits(hp):
    neurons_per_layer = []
    total_parameters = 0
    max_parameters = 8000

    if FOUR_LAYERS:
      layer_num = 4
    else:
      layer_num = 3

    for i in range(layer_num):
      neurons = hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
      neurons_per_layer.append(neurons)

    input_size = 602
    total_parameters += (
        input_size * neurons_per_layer[0] + neurons_per_layer[0]
    )

    for i in range(1, len(neurons_per_layer)):
      total_parameters += (
          neurons_per_layer[i - 1] * neurons_per_layer[i]
          + neurons_per_layer[i]
      )

    total_parameters += neurons_per_layer[-1] * 10 + 10

    if total_parameters < max_parameters:
      return None

    # Create Model
    model = Sequential()
    model.add(Input(shape=(602,), name='input1'))
    l2_rate = 1e-5#hp.Float('l2', 1e-7, 1e-4, sampling='LOG')
    if TUNER_DROPOUT:
      dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)

    for i, n in enumerate(neurons_per_layer):
      model.add(
          Dense(
              n,
              name=f'fc{i}',
              kernel_initializer='lecun_uniform',
              kernel_regularizer=l2(l2_rate),#0.001),
          )
      )
      model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
      if TUNER_DROPOUT:
        model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))

    model.add(Dense(10, name='output', kernel_initializer='lecun_uniform'))

    # Learning rate configuration
    lr_initial = hp.Float(
        'learning_rate', 1e-6, 5e-3, sampling='LOG', default=0.0001
    )
    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr_initial,
        decay_steps=LR_DECAY_STEPS,
        decay_rate=LR_DECAY_RATE,
        staircase=LR_STAIRCASE,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mean_squared_error',
        metrics=['mean_squared_error'],
        #loss='mean_absolute_error',
        #metrics = ['mean_absolute_error'],
    )
    return model'''

"if KERAS_TUNER: #USING TORCH\n\n  def build_hypermodel(hp):\n    # Hyperparameters to tune\n    if FOUR_LAYERS:\n      layer_num = 4\n    else:\n      layer_num = 3\n\n    neurons_per_layer = [\n        hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)\n        for i in range(layer_num)\n    ]\n    l2_rate = 1e-5 #hp.Float('l2', 1e-7, 1e-4, sampling='LOG')\n    if TUNER_DROPOUT:\n      dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)\n\n    # Create Model\n    model = Sequential()\n    model.add(Input(shape=(7,), name='input1'))\n\n    for i, n in enumerate(neurons_per_layer):\n      model.add(\n          Dense(\n              n,\n              name=f'fc{i}',\n              kernel_initializer='lecun_uniform',\n              kernel_regularizer=l2(l2_rate),#0.001),\n          )\n      )\n      model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))\n      if TUNER_DROPOUT:\n        model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))\n

In [89]:
#from keras_tuner import Objective
#tensorflow
if KERAS_TUNER:
    if TUNER_LIMITS:
        tuner = RandomSearch(
            build_hypermodel_with_limits,
            objective='val_loss',
            max_trials=100,
            executions_per_trial=1,
            #directory='hyper_tuning',
            directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning_sur',
            project_name='mlp_tuning',
            #overwrite=True
        )
    else:
        tuner = RandomSearch(
        build_hypermodel,
        objective='val_loss',
        max_trials=100,
        executions_per_trial=1,
        #directory='hyper_tuning',
        directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning_sur',
        project_name='mlp_tuning',
        #overwrite=True
    )
#pytorch
'''if KERAS_TUNER: #USING TORCH
    if TUNER_LIMITS:
        tuner = RandomSearch(
            build_hypermodel_with_limits,
            objective=Objective('val_mean_squared_error', direction='min'),
            max_trials=500,
            executions_per_trial=1,
            #directory='hyper_tuning_sur',
            directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',
            project_name='mlp_tuning',
           # overwrite=True
        )
        

    else:
        tuner = RandomSearch(
        build_hypermodel,
        #objective='val_loss',
        objective=Objective('val_mean_squared_error', direction='min'),
        max_trials=500,
        executions_per_trial=1,
        #directory='hyper_tuning_sur',
        directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',
        project_name='mlp_tuning',
        #overwrite=True
    )'''

Reloading Tuner from /scratch/fast/7DayLifetime/oliviagz/hyper_tuning_sur/mlp_tuning/tuner0.json


"if KERAS_TUNER: #USING TORCH\n    if TUNER_LIMITS:\n        tuner = RandomSearch(\n            build_hypermodel_with_limits,\n            objective=Objective('val_mean_squared_error', direction='min'),\n            max_trials=500,\n            executions_per_trial=1,\n            #directory='hyper_tuning_sur',\n            directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',\n            project_name='mlp_tuning',\n           # overwrite=True\n        )\n        \n\n    else:\n        tuner = RandomSearch(\n        build_hypermodel,\n        #objective='val_loss',\n        objective=Objective('val_mean_squared_error', direction='min'),\n        max_trials=500,\n        executions_per_trial=1,\n        #directory='hyper_tuning_sur',\n        directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',\n        project_name='mlp_tuning',\n        #overwrite=True\n    )"

In [92]:
if KERAS_TUNER:
    # Setup Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=TRAIN_EARLY_STOPPING_PATIENCE,
        verbose=1,
        #restore_best_weights=True
    )

In [93]:
if KERAS_TUNER:
    # Perform hyperparameter tuning
    tuner.search(np.asarray(x_train_val), 
                 np.asarray(lin_log_Y_train), 
                 epochs=50, 
                 batch_size= 32,#TRAIN_BATCH_SIZE, 
                 validation_split=TRAIN_VALIDATION_SPLIT, 
                 callbacks=[early_stopping], 
                 verbose=1)

Epoch 1/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 6.5972 - mean_squared_error: 0.1468 - val_loss: 6.4921 - val_mean_squared_error: 0.0825
Epoch 2/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.4638 - mean_squared_error: 0.0956 - val_loss: 6.3858 - val_mean_squared_error: 0.0602
Epoch 3/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.3631 - mean_squared_error: 0.0792 - val_loss: 6.2930 - val_mean_squared_error: 0.0517
Epoch 4/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.2709 - mean_squared_error: 0.0710 - val_loss: 6.2050 - val_mean_squared_error: 0.0472
Epoch 5/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.1823 - mean_squared_error: 0.0652 - val_loss: 6.1198 - val_mean_squared_error: 0.0441
Epoch 6/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.0961 - mean_squared_error: 0.0603 - val_loss: 6.0363 - val_mean_squared_error: 0.0411
Epoch 7/50
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.0122 - mean_squared_error: 0.0563 - val_los

KeyboardInterrupt: 

In [ ]:
if KERAS_TUNER:
    model = tuner.get_best_models(num_models=1)[0]
    print(model) 

In [20]:

if KERAS_TUNER:
  best_model_file = 'model_sur/best_keras_model.keras'

  # Retrieve the best model from the tuner
  model = tuner.get_best_models(num_models=1)[0]

  # Save and reload it to ensure standalone functionality
  model.save(best_model_file)
  model = load_model(best_model_file)

# Re-compile to build the metrics structure cleanly
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_squared_error'],
    
)

I0000 00:00:1787755919.051756    2116 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/oliviag/.conda/envs/spice_env/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:868: UserWarning: Skipping variable loading for optimizer 'adam', because it has 17 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### View the model

In [21]:
#if this gives an error do 'conda activate cryo-modelling-env' and 'conda install -c conda-forge python-graphviz'
#tf.keras.utils.plot_model(model, to_file='{}_model.png'.format(model_shape), show_shapes=True, show_layer_names=True)
model.summary()
with open('model_summary.txt', 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc0 (Dense)                     │ (None, 2016)           │        16,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 2016)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 2016)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 1248)           │     2,517,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 1248)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 1248)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 1344)           │     1,678,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 1344)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 1344)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 370)            │       497,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,709,650 (17.97 MB)

 Trainable params: 4,709,650 (17.97 MB)

 Non-trainable params: 0 (0.00 B)

### Evaluation

Although we may plot and print many metrics, we focus only on **Mean Squared Error (MSE).**

Plot training history.

In [22]:
#%matplotlib ipympl
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [25]:
test_loss_result = model.evaluate(np.asarray(x_test), np.asarray(lin_log_Y_test))[0]
print(x_test[0])
print(test_loss_result)

337/337 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 6.5980e-04 - mean_squared_error: 2.1520e-04
[0.05402498 0.96131346 0.36993976 0.48437063 0.40750882 0.24778849
 0.54218151]
0.0006598001345992088


In [26]:
print('Current loss {}: {}'.format(TRAIN_LOSS, test_loss_result))

Current loss mean_squared_error: 0.0006598001345992088


In [27]:
import csv
if 'model' in locals() or 'model' in globals():
    model_shape = "-".join([str(layer.units) for layer in model.layers if hasattr(layer, 'units')])
else:
    model_shape = "Unknown_Shape"
csv_data = [[
    PARAM,
    DATA_AUGMENTATION,
    model_shape,
    test_loss_result,
    TRAIN_LOSS,
    TRAIN_DROPOUT_RATE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_BATCH_SIZE,
    TRAIN_VALIDATION_SPLIT,
    LR_INITIAL,
    LR_DECAY_STEPS,
    LR_DECAY_RATE,
    LR_STAIRCASE
    ]]
TRAIN_BATCH_SIZE = 256
csv_file = 'history_losses.csv'  #this doesnt reqrite this file so you need to delete this if you want something fresh

if not os.path.exists(csv_file):
    with open(csv_file, 'w') as file:
        file.write('data_param,data_augmentation,model_shape,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case\n')
with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(csv_data)

# Convert data to DataFrame for easier display
df = pd.read_csv(csv_file)

def color_red_column(s):
    return ['color: red' if v else '' for v in s]

styled_df = df.style.apply(color_red_column, subset=['test_loss'])

# Display the DataFrame as a table
display(styled_df)
#qgrid_widget = qgrid.show_grid(df, show_toolbar=True)

,data_param,data_augmentation,model_shape,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case
0,iv,True,4000-500-4600-7,0.062447,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
1,iv,True,4000-500-4600-7,0.062447,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
2,iv,True,4000-500-4600-7,0.062447,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
3,iv,True,4000-500-4600-7,0.062447,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
4,iv,True,3200-2700-3400-7,0.062326,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
5,iv,True,4100-600-2300-7,0.062052,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
6,iv,True,1300-700-5000-7,0.061732,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
7,iv,True,3700-600-600-7,0.062580,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
8,iv,True,1000-1600-4100-7,0.063081,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False
9,iv,True,2100-3000-1100-7,0.063363,mean_squared_error,0,50,32,0.200000,0.000100,100,0.990000,False


In [28]:
y_pred = model.predict(np.array(x_test))#(lin_log_X_test)

337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


### Scaled Test vs Predictions if Y is scaled

In [29]:
if SCALE_Y:
    paramNames = ['vth0_nom','vsat_nom', 'rdsw_nom','u0_nom','nfactor_nom','eta0_nom','delta_nom']
    x_test_arr = np.asarray(x_test)  # Shape: (N_samples, 10)
    y_test_arr = np.asarray(lin_log_Y_test)  # Shape: (N_samples, N_iv_points)
    y_pred_arr = np.asarray(y_pred)
    pow2_errors = (y_test_arr - y_pred_arr) ** 2
    abs_errors = np.abs(y_test_arr - y_pred_arr)
    for i in range(min(2, len(y_test_arr))):
        print(f"==================== TEST SAMPLE #{i+1} ====================")
    
        # 1. Print Input Parameters (X)
        print("--- Input SPICE Parameters ---")
        for l in range(min(len(paramNames), x_test_arr.shape[1])):
          print(f"  {paramNames[l]:<6}: {x_test_arr[i][l]:12.6e}")
    
        # 2. Print IV Curve Prediction Performance (Y)
        mean_abs_err = np.mean(abs_errors[i])
        max_abs_err = np.max(abs_errors[i])
        rmse = np.sqrt(np.mean(pow2_errors[i]))
    
        print("\n--- Predicted IV Curve Error Summary ---")
        print(f"  Total Voltage Bias Points: {len(y_test_arr[i])}")
        print(f"  Root Mean Sq Error (RMSE): {rmse:12.6e}")
        print(f"  Mean Absolute Error (MAE): {mean_abs_err:12.6e}")
        print(f"  Max Absolute Error       : {max_abs_err:12.6e}")
    
        # 3. Print first 5 IV data points as a sample comparison
        print("\n  Sample IV Points (First 5):")
        for pt in range(min(5, len(y_test_arr[i]))):
          print(
              f"    Pt #{pt+1}: Ref = {y_test_arr[i][pt]:10.6e} | Pred ="
              f" {y_pred_arr[i][pt]:10.6e} | AbsEr = {abs_errors[i][pt]:10.6e}"
          )
        print("\n")
    
    
    #pow2_errors = [(y_test[i]-y_pred[i][0])**2 for i in range(len(y_pred))]
    #abs_errors = [abs(y_test[i]-y_pred[i][0]) for i in range(len(y_pred))]
    #min=1
    #for i in range(2): 
        #if abs_errors[i].any()<min: 
            #min = abs_errors[i]
        #print("IV curve #: " + str(i+1))
        #for l in range(len(y_test[i])):
            #print('Param: {}, Ref: {}, Pred: {}, Pow2Er: {}, AbsEr: {}'.format(paramNames[l],y_test[i][l], y_pred[i][l], pow2_errors[i][l], abs_errors[i][l]))
        #print("")

==================== TEST SAMPLE #1 ====================
--- Input SPICE Parameters ---
  vth0_nom: 5.402498e-02
  vsat_nom: 9.613135e-01
  rdsw_nom: 3.699398e-01
  u0_nom: 4.843706e-01
  nfactor_nom: 4.075088e-01
  eta0_nom: 2.477885e-01
  delta_nom: 5.421815e-01

--- Predicted IV Curve Error Summary ---
  Total Voltage Bias Points: 370
  Root Mean Sq Error (RMSE): 1.477874e-02
  Mean Absolute Error (MAE): 9.804004e-03
  Max Absolute Error       : 9.746489e-02

  Sample IV Points (First 5):
    Pt #1: Ref = 0.000000e+00 | Pred = -3.123539e-04 | AbsEr = 3.123539e-04
    Pt #2: Ref = 0.000000e+00 | Pred = -3.030994e-04 | AbsEr = 3.030994e-04
    Pt #3: Ref = 0.000000e+00 | Pred = -1.076779e-03 | AbsEr = 1.076779e-03
    Pt #4: Ref = 0.000000e+00 | Pred = 8.395931e-05 | AbsEr = 8.395931e-05
    Pt #5: Ref = 0.000000e+00 | Pred = -7.434436e-04 | AbsEr = 7.434436e-04


==================== TEST SAMPLE #2 ====================
--- Input SPICE Parameters ---
  vth0_nom: 8.519510e-01
  vsat_no

### Unscaled Test vs Predictions

In [31]:
import joblib
paramNames = [
    'vth0_nom',
    'vsat_nom',
    'rdsw_nom',
    'u0_nom',
    'nfactor_nom',
    'eta0_nom',
    'delta_nom',
]

# Ensure NumPy arrays float32/float64
y_test_arr = np.asarray(lin_log_Y_test, dtype=np.float64)
y_pred_arr = np.asarray(y_pred, dtype=np.float64)

if SCALE_Y:
  print("Unscaling y_test and y_pred...")
  # 1. Load scalers ONCE into memory outside the loop
  scalers = [
      joblib.load(f'scalers/scaler_y_{name}.save') for name in paramNames
  ]

  # 2. Vectorized unscaling column by column
  y_test_unscaled = np.zeros_like(y_test_arr)
  y_pred_unscaled = np.zeros_like(y_pred_arr)

  for col_idx, scaler in enumerate(scalers):
    # Inverse transform full columns at once
    y_test_unscaled[:, col_idx] = scaler.inverse_transform(
        y_test_arr[:, col_idx].reshape(-1, 1)
    ).ravel()
    y_pred_unscaled[:, col_idx] = scaler.inverse_transform(
        y_pred_arr[:, col_idx].reshape(-1, 1)
    ).ravel()

  print('y_test_unscaled[0]: ', y_test_unscaled[0])
  print('y_pred_unscaled[0]: ', y_pred_unscaled[0])
else:
  y_test_unscaled = y_test_arr
  y_pred_unscaled = y_pred_arr

# 3. Compute element-wise errors vectorized shape: (N_samples, 7)
pow2_errors = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors = np.abs(y_test_unscaled - y_pred_unscaled)

# 4. Print results for the first 2 samples cleanly
num_samples_to_print = min(2, len(y_test_unscaled))

for i in range(num_samples_to_print):
  print(f"--- Parameter Set / IV Curve #: {i+1} ---")
  for l in range(min(len(paramNames), y_test_unscaled.shape[1])):
    print(
        f"Param: {paramNames[l]:<12} | Ref: {y_test_unscaled[i][l]:12.6e} |"
        f" Pred: {y_pred_unscaled[i][l]:12.6e} | Pow2Er:"
        f" {pow2_errors[i][l]:12.6e} | AbsEr: {abs_errors[i][l]:12.6e}"
    )
  print("")

Unscaling y_test and y_pred...
y_test_unscaled[0]:  [1.00043475e-01 1.00040627e-01 2.00005299e-01 6.90571355e-01
 1.00237019e-01 1.00138920e-01 5.70892835e-04 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.

## Compare predictions vs. test set

In [32]:
%matplotlib ipympl
#import matplotlib.pyplot as plt
import matplotlib.pyplot as plt


import matplotlib.cm as cm

In [34]:
import numpy as np

paramNames = [
    'vth0_nom',
    'vsat_nom',
    'rdsw_nom',
    'u0_nom',
    'nfactor_nom',
    'eta0_nom',
    'delta_nom',
]

print("=== DATASET ORIENTATION DIAGNOSTIC ===")
print("X_test shape:", np.shape(x_test))  # Should be (2000, 7)
print("y_test shape:", np.shape(lin_log_Y_test))  # Should be (2000, N_points)

# 1. Inspect Sample 0 Input Parameters (X)
print("\n--- Input Parameters (X_test[0]) ---")
sample_x = x_test[0]
for idx, val in enumerate(sample_x):
  name = paramNames[idx] if idx < len(paramNames) else f"Param_{idx}"
  print(f"  [{idx}] {name:<12}: {val:12.6e}")

# 2. Inspect Sample 0 Target IV Curve (Y)
print("\n--- Target IV Curve (y_test[0]) ---")
sample_y = np.asarray(lin_log_Y_test[0])
print(f"  IV Points count : {len(sample_y)}")
print(f"  Min Target Value: {np.min(sample_y):12.6e}")
print(f"  Max Target Value: {np.max(sample_y):12.6e}")

# Check for NaNs/Infs in targets
if np.isnan(sample_y).any():
  print("Found NaN values in y_test!")
if np.isinf(sample_y).any():
  print("Found Inf values in y_test!")

=== DATASET ORIENTATION DIAGNOSTIC ===
X_test shape: (10781, 7)
y_test shape: (10781, 370)

--- Input Parameters (X_test[0]) ---
  [0] vth0_nom    : 5.402498e-02
  [1] vsat_nom    : 9.613135e-01
  [2] rdsw_nom    : 3.699398e-01
  [3] u0_nom      : 4.843706e-01
  [4] nfactor_nom : 4.075088e-01
  [5] eta0_nom    : 2.477885e-01
  [6] delta_nom   : 5.421815e-01

--- Target IV Curve (y_test[0]) ---
  IV Points count : 370
  Min Target Value: 0.000000e+00
  Max Target Value: 9.172844e-01


In [37]:
print(x_test[0])

[0.05402498 0.96131346 0.36993976 0.48437063 0.40750882 0.24778849
 0.54218151]
